<a href="https://colab.research.google.com/github/JSJeong-me/Retriever/blob/main/Phishing/Voice-Phishing01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss-cpu transformers

In [ ]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# JSON 파일 로드
json_file_path = "/content/law_enforcement_scam_0228.json"

with open(json_file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 텍스트 데이터 추출
texts = [entry["text"] for entry in data]
ids = [entry["id"] for entry in data]

# SentenceTransformer 모델 로드 (BERT 기반 임베딩)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 텍스트 데이터 임베딩 생성
embeddings = model.encode(texts, convert_to_numpy=True)

# FAISS Index 생성 (L2 거리 기반)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# ID와 매핑
id_map = {i: ids[i] for i in range(len(ids))}

# 검색 함수
def search(query, top_k=5):
    query_vector = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vector, top_k)

    results = []
    for i in range(top_k):
        idx = indices[0][i]
        results.append({"id": id_map[idx], "text": texts[idx], "distance": distances[0][i]})

    return results

# 예제 검색 실행
query = "계좌 개설 여부 확인"
results = search(query, top_k=3)

# 검색 결과 출력
for res in results:
    print(f"ID: {res['id']}, Text: {res['text']}, Distance: {res['distance']:.4f}")
